# Ex2 — NLP 67658

In [ ]:
# Shared imports + reproducibility + hyperparameters + helpers
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

from ex2 import get_data, category_dict, transformer_classification
categories = list(category_dict.keys())

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available()
                      else ('mps' if torch.backends.mps.is_available() else 'cpu'))

# Q1/Q2 hyperparameters
epochs = 20
batch_size = 16
lr = 1e-3
feature_dim = 2000
hidden_dim = 500
num_labels = len(category_dict)

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_labels),
        )
    def forward(self, x):
        return self.net(x)

def train_and_eval(model, train_loader, test_loader, optimizer, criterion, epochs, device):
    train_losses, val_accs = [], []
    for epoch in range(1, epochs + 1):
        model.train()
        running, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * xb.size(0)
            n += xb.size(0)
        train_losses.append(running / n)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                correct += (model(xb).argmax(1) == yb).sum().item()
                total += yb.size(0)
        val_accs.append(correct / total)
        print(f'Epoch {epoch:02d} | train loss {train_losses[-1]:.4f} | val acc {val_accs[-1]:.4f}')
    return train_losses, val_accs

print(f'device: {device}')


# Q1 — Log-linear classifier (TFIDF + single-layer perceptron)

## Q1 (a) portion=0.1

In [ ]:
# Data + TFIDF (portion=0.1)
portion = 0.1
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train log-linear (portion via cell above)
model = nn.Linear(feature_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q1_train_losses_a, q1_val_accs_a = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_train_losses_a) + 1), q1_train_losses_a, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q1 Log-linear (portion=0.1) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_val_accs_a) + 1), q1_val_accs_a, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q1 Log-linear (portion=0.1) — Val accuracy')
plt.tight_layout(); plt.show()


## Q1 (b) portion=0.2

In [ ]:
# Data + TFIDF (portion=0.2)
portion = 0.2
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train log-linear (portion via cell above)
model = nn.Linear(feature_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q1_train_losses_b, q1_val_accs_b = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_train_losses_b) + 1), q1_train_losses_b, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q1 Log-linear (portion=0.2) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_val_accs_b) + 1), q1_val_accs_b, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q1 Log-linear (portion=0.2) — Val accuracy')
plt.tight_layout(); plt.show()


## Q1 (c) portion=0.5

In [ ]:
# Data + TFIDF (portion=0.5)
portion = 0.5
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train log-linear (portion via cell above)
model = nn.Linear(feature_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q1_train_losses_c, q1_val_accs_c = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_train_losses_c) + 1), q1_train_losses_c, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q1 Log-linear (portion=0.5) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_val_accs_c) + 1), q1_val_accs_c, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q1 Log-linear (portion=0.5) — Val accuracy')
plt.tight_layout(); plt.show()


## Q1 (d) portion=1.0

In [ ]:
# Data + TFIDF (portion=1.0)
portion = 1.0
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train log-linear (portion via cell above)
model = nn.Linear(feature_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q1_train_losses_d, q1_val_accs_d = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_train_losses_d) + 1), q1_train_losses_d, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q1 Log-linear (portion=1.0) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q1_val_accs_d) + 1), q1_val_accs_d, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q1 Log-linear (portion=1.0) — Val accuracy')
plt.tight_layout(); plt.show()


# Q2 — MLP classifier (TFIDF + one hidden layer, dim=500)

## Q2 (a) portion=0.1

In [ ]:
# Data + TFIDF (portion=0.1)
portion = 0.1
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train MLP (portion via cell above)
model = MLP(feature_dim, hidden_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q2_train_losses_a, q2_val_accs_a = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_train_losses_a) + 1), q2_train_losses_a, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q2 MLP (portion=0.1) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_val_accs_a) + 1), q2_val_accs_a, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q2 MLP (portion=0.1) — Val accuracy')
plt.tight_layout(); plt.show()


## Q2 (b) portion=0.2

In [ ]:
# Data + TFIDF (portion=0.2)
portion = 0.2
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train MLP (portion via cell above)
model = MLP(feature_dim, hidden_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q2_train_losses_b, q2_val_accs_b = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_train_losses_b) + 1), q2_train_losses_b, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q2 MLP (portion=0.2) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_val_accs_b) + 1), q2_val_accs_b, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q2 MLP (portion=0.2) — Val accuracy')
plt.tight_layout(); plt.show()


## Q2 (c) portion=0.5

In [ ]:
# Data + TFIDF (portion=0.5)
portion = 0.5
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train MLP (portion via cell above)
model = MLP(feature_dim, hidden_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q2_train_losses_c, q2_val_accs_c = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_train_losses_c) + 1), q2_train_losses_c, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q2 MLP (portion=0.5) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_val_accs_c) + 1), q2_val_accs_c, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q2 MLP (portion=0.5) — Val accuracy')
plt.tight_layout(); plt.show()


## Q2 (d) portion=1.0

In [ ]:
# Data + TFIDF (portion=1.0)
portion = 1.0
x_train, y_train, x_test, y_test = get_data(categories=categories, portion=portion)

vectorizer = TfidfVectorizer(max_features=feature_dim)
X_train = vectorizer.fit_transform(x_train).toarray().astype(np.float32)
X_test  = vectorizer.transform(x_test).toarray().astype(np.float32)
y_train = np.array(y_train, dtype=np.int64)
y_test  = np.array(y_test,  dtype=np.int64)

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test)),
                          batch_size=batch_size)

print(f'train size: {len(x_train)}, test size: {len(x_test)}')


In [ ]:
# Train MLP (portion via cell above)
model = MLP(feature_dim, hidden_dim, num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

q2_train_losses_d, q2_val_accs_d = train_and_eval(
    model, train_loader, test_loader, optimizer, criterion, epochs, device)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_train_losses_d) + 1), q2_train_losses_d, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q2 MLP (portion=1.0) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q2_val_accs_d) + 1), q2_val_accs_d, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q2 MLP (portion=1.0) — Val accuracy')
plt.tight_layout(); plt.show()


# Q3 — Fine-tune distilroberta-base (uses `transformer_classification` from ex2.py)

## Q3 (a) portion=0.1

In [ ]:
# Q3 fine-tune (portion=0.1) — delegates to ex2.transformer_classification
q3_train_losses_a, q3_val_accs_a = transformer_classification(portion=0.1)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q3_train_losses_a) + 1), q3_train_losses_a, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q3 distilroberta (portion=0.1) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q3_val_accs_a) + 1), q3_val_accs_a, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q3 distilroberta (portion=0.1) — Val accuracy')
plt.tight_layout(); plt.show()


## Q3 (b) portion=0.2

In [ ]:
# Q3 fine-tune (portion=0.2) — delegates to ex2.transformer_classification
q3_train_losses_b, q3_val_accs_b = transformer_classification(portion=0.2)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q3_train_losses_b) + 1), q3_train_losses_b, marker='o')
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('Q3 distilroberta (portion=0.2) — Train loss')
plt.tight_layout(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(q3_val_accs_b) + 1), q3_val_accs_b, marker='o', color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation accuracy')
ax.set_title('Q3 distilroberta (portion=0.2) — Val accuracy')
plt.tight_layout(); plt.show()
